# 22 - Crop gate: "is this a maize, tomato or potato leaf at all?"

**Why:** notebook 21d sent 1,200 leaves of other crops and other tomato diseases through the
deployed system. With the crop chosen as tomato 3.3% got a diagnosis, as potato 8.7% (bean
leaves as potato 15.3%), all above 0.957 confidence, so the threshold cannot catch them: the
disease model has no "other crop" class. Notebook 21d also showed the disease model is as unsure
of a real tomato's crop as of a bean's, so its own probabilities cannot be used as a gate.

**Design (decided 3 Nov 2026, before anything was scored), built exactly like the leaf check
(notebook 19, `src/leaf_check.py`):** the leaf check's frozen MobileNetV3 turns the photo into a
summary, and one logistic-regression layer answers "our crops or another crop". It runs after the
leaf check and before the disease model; the disease model, calibration and threshold do not change.

| Use | Our crops (maize, tomato, potato) | Other crops |
|---|---|---|
| train the layer | the 5,000 training-leaf summaries notebook 19 saved | apple, blueberry, cherry, pepper, raspberry, strawberry (PlantVillage and PlantDoc) |
| set the cut-off | the validation split | - |
| **decide (rule)** | **selection set** (field photos from sources never trained on) | **crops the layer never saw:** beans (all of iBean, Uganda), grape, soybean, squash, orange, peach |
| report only | test split, PlantDoc | PlantDoc's other tomato diseases (these ARE tomato: passing them is correct) |

**Rule (fixed before running):**

- Cut-off: the highest probability at which at least **99% of validation leaves of our crops pass**
  (as the leaf check: blocking a real farmer costs more than letting an odd photo through).
- The gate is used only if, at that cut-off, it lets **at least 97% of selection-set photos**
  through (new field photos of our crops) **and** refuses **at least 80% of the unseen crops**.
  Otherwise it is reported and stays off.
- Reported, not used to decide: notebook 21d's outcomes before and after the gate, on the
  unseen crops only (21d photos of the training crops are marked as seen).

A GPU runtime is best (T4); about 30-40 minutes, most of it downloads. Run notebook 19 and 21d
first: their saved summaries and decisions are read from Drive.

In [1]:
from pathlib import Path
import sys

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/BilliartMuthoni/confidence-calibrated-crop-diagnosis.git"
REPO_DIR = Path("/content/drive/MyDrive/Sproutly-app")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

ML_DIR = REPO_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

import config

Mounted at /content/drive


In [2]:
!git -C {REPO_DIR} config core.fileMode false
!git -C {REPO_DIR} pull --ff-only
!git -C {REPO_DIR} status --short --branch

Already up to date.
Refresh index: 100% (541/541), done.
## main...origin/main
 M ml/notebooks/22_crop_gate.ipynb
?? ml/results/fusion_quick_test/


In [3]:
%pip install -q -r {ML_DIR}/requirements.txt

## Settings, and what notebooks 19 and 21d saved

In [4]:
import importlib
import json
import shutil

import numpy as np
import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

from src import leaf_check, training
from src.data import d17, image_dataset, out_of_scope
for module in (leaf_check, training, d17, image_dataset, out_of_scope):
    importlib.reload(module)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = config.RANDOM_SEED
ARCHITECTURE = "mobilenet_v3_large"           # the leaf check's network, so the backend can share it
TRAIN_CROPS = {"apple", "blueberry", "cherry", "pepper", "raspberry", "strawberry"}
UNSEEN_CROPS = {"grape", "soybean", "squash", "orange", "peach"}   # plus beans (iBean)
PASS_RATE, SELECTION_PASS, UNSEEN_REJECT = 0.99, 0.97, 0.80
assert not TRAIN_CROPS & UNSEEN_CROPS

OUT_DIR = config.MODELS_DIR / "crop_gate"
REPO_RESULTS = ML_DIR / "results" / "crop_gate"
LOCAL = Path("/content/crop_gate")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RERUN_REASON = ""
if (REPO_RESULTS / "decision.json").exists() or (OUT_DIR / "decision.json").exists():
    assert RERUN_REASON.strip(), "Already decided once. Write RERUN_REASON to run it again."

saved = config.MODELS_DIR / "leaf_check" / f"{ARCHITECTURE}_features.npz"
assert saved.exists(), "Notebook 19's saved summaries are missing from Drive."
OURS = dict(np.load(saved, allow_pickle=True))
selection_rows = pd.read_csv(config.SPLITS_DIR / "selection.csv")
assert len(selection_rows) == len(OURS["leaf_selection"]), "selection.csv changed since notebook 19"
oos_dir = config.MODELS_DIR / "out_of_scope"
assert (oos_dir / "decisions.csv").exists(), "Run notebook 21d first."
print({k: v.shape for k, v in OURS.items() if k.startswith("leaf_")}, DEVICE)

{'leaf_train': (4999, 960), 'leaf_val': (2385, 960), 'leaf_test': (2441, 960), 'leaf_plantdoc': (953, 960), 'leaf_selection': (3821, 960)} cuda


## Other-crop photos

Training crops and unseen crops are downloaded separately; no crop is in both. Beans (iBean)
are all unseen.

In [5]:
def tag(frame, use):
    crops = frame["class"].map(out_of_scope.crop_name) if "ibean" not in frame["source"].iat[0] else "bean"
    return frame.assign(crop=crops, use=use)


OTHERS = pd.concat([
    tag(out_of_scope.plantvillage_other_crops(LOCAL / "pv_train", 1500, SEED, TRAIN_CROPS), "train"),
    tag(out_of_scope.plantdoc_other_crops(LOCAL / "pd_train", 1000, SEED, TRAIN_CROPS), "train"),
    tag(out_of_scope.ibean(LOCAL / "ibean", 100_000, SEED), "unseen"),
    tag(out_of_scope.plantvillage_other_crops(LOCAL / "pv_unseen", 1000, SEED, UNSEEN_CROPS), "unseen"),
    tag(out_of_scope.plantdoc_other_crops(LOCAL / "pd_unseen", 1000, SEED, UNSEEN_CROPS), "unseen"),
], ignore_index=True)


def opens(path):
    try:
        with Image.open(path) as image:
            image.convert("RGB")
        return True
    except Exception:
        return False


readable = OTHERS["path"].map(opens)
print(f"{(~readable).sum()} unreadable files dropped")
OTHERS = OTHERS[readable].reset_index(drop=True)
assert set(OTHERS.loc[OTHERS["use"] == "train", "crop"]) <= TRAIN_CROPS
assert not set(OTHERS.loc[OTHERS["use"] == "unseen", "crop"]) & TRAIN_CROPS
TOMATO_OTHER = out_of_scope.plantdoc_other_tomato_diseases(config.SPLITS_DIR / "unseen_test.csv",
                                                           config.RAW_DATA_DIR, 100_000, SEED)
display(OTHERS.groupby(["use", "source", "crop"]).size().rename("photos").to_frame())
print(len(TOMATO_OTHER), "PlantDoc tomato leaves with other diseases (report only)")

data/train.zip: reconstructing file:   0%|          |  0.00B /  144MB            

data/train.zip: downloading bytes:           |  0.00B            

data/validation.zip: reconstructing file:   0%|          |  0.00B / 18.5MB            

data/validation.zip: downloading bytes:           |  0.00B            

data/test.zip: reconstructing file:   0%|          |  0.00B / 17.7MB            

data/test.zip: downloading bytes:           |  0.00B            

0 unreadable files dropped


photos
use    source                   crop              
train  plantdoc_other_crops     apple          273
                                blueberry      117
                                cherry          57
                                pepper         132
                                raspberry      119
                                strawberry      96
       plantvillage_other_crops apple          500
                                blueberry      125
                                cherry         250
                                pepper         250
                                raspberry      125
                                strawberry     250
unseen ibean_bean_field         bean          1295
       plantdoc_other_crops     grape          133
                                peach          112
                                soybean         65
                                squash         130
       plantvillage_other_crops grape          444
                                orange         112
                                peach          222
                                soybean        111
                                squash         111

371 PlantDoc tomato leaves with other diseases (report only)


## STOP: look at the photos

One sheet per group. Check that the training and unseen sheets show **leaves of other crops**
(fruit or drawings in PlantDoc are expected; the leaf check stops most of those in the app) and
**no maize, tomato or potato leaf**.

In [6]:
REVIEW_DIR = LOCAL / "review"
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
sheets = []
for (use, source), part in OTHERS.groupby(["use", "source"]):
    paths = [Path(p) for p in part.sample(min(36, len(part)), random_state=SEED)["path"]]
    sheet = d17.contact_sheet(paths, f"{use} | {source} | {len(part)} photos | {', '.join(sorted(part['crop'].unique()))}")
    sheet.save(REVIEW_DIR / f"{use}__{source}.jpg", quality=85)
    sheets.append(REVIEW_DIR / f"{use}__{source}.jpg")
    display(sheet)

Output hidden; open in https://colab.research.google.com to view.

In [9]:
# After looking at EVERY sheet above, change False to True and run this cell.
I_CHECKED_THE_PHOTOS = True

assert I_CHECKED_THE_PHOTOS, "Look at every sheet above first, then set I_CHECKED_THE_PHOTOS = True."
print("Checked.")

Checked.


## Summaries from the leaf check's frozen network

Each photo is first shrunk to a shorter side of 256 px, exactly as notebook 19's leaf photos
were, so both sides of the layer see the same kind of image.

In [10]:
def shrunk(paths, folder):
    out = []
    for i, p in enumerate(tqdm(list(paths), desc=f"Shrinking {folder}")):
        target = LOCAL / "shrunk" / folder / f"{i:05d}.jpg"
        image_dataset._cache_one(Path(p), target)
        out.append(target)
    return out


network = leaf_check.backbone(ARCHITECTURE)


def summaries(paths, folder):
    x, ok = leaf_check.features(network, shrunk(paths, folder), DEVICE)
    assert ok.all(), f"{(~ok).sum()} unreadable photos in {folder}"
    return x


OTHER_X = summaries(OTHERS["path"], "others")
TOMATO_X = summaries(TOMATO_OTHER["path"], "tomato_other")
np.savez(OUT_DIR / "other_features.npz", others=OTHER_X, tomato_other=TOMATO_X)
print(OTHER_X.shape, TOMATO_X.shape)

Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-5c1a4163.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-5c1a4163.pth


100%|██████████| 21.1M/21.1M [00:00<00:00, 224MB/s]


Shrinking others:   0%|          | 0/5029 [00:00<?, ?it/s]

Shrinking tomato_other:   0%|          | 0/371 [00:00<?, ?it/s]

(5029, 960) (371, 960)


## Fit the layer, set the cut-off, apply the rule

In [11]:
train = (OTHERS["use"] == "train").to_numpy()
head = leaf_check.fit(OURS["leaf_train"], OTHER_X[train], SEED)
prob = lambda x: leaf_check.leaf_probability(head, x)   # probability "one of our crops"
CUT_OFF = leaf_check.threshold_for_pass_rate(prob(OURS["leaf_val"]), PASS_RATE)

unseen = ~train
selection_pass = float((prob(OURS["leaf_selection"]) >= CUT_OFF).mean())
unseen_reject = float((prob(OTHER_X[unseen]) < CUT_OFF).mean())
ADOPTED = selection_pass >= SELECTION_PASS and unseen_reject >= UNSEEN_REJECT

rows = [
    ("our crops: validation (sets the cut-off)", OURS["leaf_val"], True),
    ("our crops: SELECTION SET (rule: >= 97% pass)", OURS["leaf_selection"], True),
    ("our crops: test split", OURS["leaf_test"], True),
    ("our crops: PlantDoc (web photos)", OURS["leaf_plantdoc"], True),
    ("other crops: UNSEEN (rule: >= 80% refused)", OTHER_X[unseen], False),
    ("other crops: training crops (in-sample)", OTHER_X[train], False),
    ("tomato with other diseases (PlantDoc; is tomato)", TOMATO_X, True),
]
report = pd.DataFrame([{"set": name, "photos": len(x), "our crop": ours,
                        "passed": float((prob(x) >= CUT_OFF).mean())} for name, x, ours in rows]).set_index("set")
report["correct"] = np.where(report["our crop"], report["passed"], 1 - report["passed"])
display(report.round(4))

by_crop = (OTHERS[unseen].assign(refused=prob(OTHER_X[unseen]) < CUT_OFF)
           .groupby(["source", "crop"])["refused"].agg(photos="size", refused="mean"))
display(by_crop.round(4))
by_source = (selection_rows.assign(passed=prob(OURS["leaf_selection"]) >= CUT_OFF)
             .groupby(["source", "class"])["passed"].agg(photos="size", passed="mean"))
display(by_source.round(4))
print(f"Cut-off {CUT_OFF:.4f} | selection passed {selection_pass:.2%} (needs {SELECTION_PASS:.0%}) | "
      f"unseen crops refused {unseen_reject:.2%} (needs {UNSEEN_REJECT:.0%}) -> "
      f"{'ADOPTED' if ADOPTED else 'NOT adopted: reported, stays off'}")

,photos,our crop,passed,correct
set,,,,
our crops: validation (sets the cut-off),2385,True,0.9904,0.9904
our crops: SELECTION SET (rule: >= 97% pass),3821,True,0.9267,0.9267
our crops: test split,2441,True,0.9832,0.9832
our crops: PlantDoc (web photos),953,True,0.7051,0.7051
other crops: UNSEEN (rule: >= 80% refused),2735,False,0.4808,0.5192
other crops: training crops (in-sample),2294,False,0.0000,1.0000
tomato with other diseases (PlantDoc; is tomato),371,True,0.4771,0.4771


photos  refused
source                   crop                    
ibean_bean_field         bean       1295   0.2347
plantdoc_other_crops     grape       133   0.8647
                         peach       112   0.9732
                         soybean      65   0.6923
                         squash      130   0.4846
plantvillage_other_crops grape       444   0.7883
                         orange      112   0.8839
                         peach       222   0.8829
                         soybean     111   0.5766
                         squash      111   0.6757

photos  \
source          class                                                        
maize_jhenaidah Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot     554   
                Corn_(maize)___healthy                                 359   
maize_natore    Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot     826   
                Corn_(maize)___Common_rust_                            126   
                Corn_(maize)___healthy                                 602   
potato_holeta   Potato___Late_blight                                    67   
                Potato___healthy                                       344   
tomato_bd       Tomato___Bacterial_spot                                 97   
                Tomato___Early_blight                                  447   
                Tomato___Late_blight                                    92   
                Tomato___healthy                                       307   

                                                                    passed  
source          class                                                       
maize_jhenaidah Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot  0.9621  
                Corn_(maize)___healthy                              1.0000  
maize_natore    Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot  0.9782  
                Corn_(maize)___Common_rust_                         0.9762  
                Corn_(maize)___healthy                              0.9751  
potato_holeta   Potato___Late_blight                                0.9104  
                Potato___healthy                                    0.6395  
tomato_bd       Tomato___Bacterial_spot                             0.9278  
                Tomato___Early_blight                               0.8971  
                Tomato___Late_blight                                0.9565  
                Tomato___healthy                                    0.8827

Cut-off 0.5021 | selection passed 92.67% (needs 97%) | unseen crops refused 51.92% (needs 80%) -> NOT adopted: reported, stays off


## Report only: notebook 21d before and after the gate

Notebook 21d's exact photos are downloaded again (same sources, seed and commit, so the same
sample) and checked against 21d's saved list. The gate runs after the blur and leaf checks, so
only photos that reached the disease model can change. Photos of the gate's training crops are
marked as seen; the honest comparison is on the unseen ones.

In [ ]:
again = pd.concat([
    out_of_scope.ibean(LOCAL / "ibean", 300, SEED),
    out_of_scope.plantvillage_other_crops(LOCAL / "pv_21d", 300, SEED),
    out_of_scope.plantdoc_other_crops(LOCAL / "pd_21d", 300, SEED),
    out_of_scope.plantdoc_other_tomato_diseases(config.SPLITS_DIR / "unseen_test.csv", config.RAW_DATA_DIR, 300, SEED),
], ignore_index=True)
then = pd.read_csv(oos_dir / "photos.csv")
assert list(again["class"]) == list(then["class"]) and list(again["source"]) == list(then["source"]), \
    "Could not rebuild notebook 21d's exact sample"
gate_pass = prob(summaries(again["path"], "photos_21d")) >= CUT_OFF

decisions = pd.read_csv(oos_dir / "decisions.csv")
crop_of_photo = np.array(["bean" if s.startswith("ibean") else "tomato" if "tomato_diseases" in s
                          else out_of_scope.crop_name(c) for s, c in zip(again["source"], again["class"])])
decisions["gate"] = np.where(np.isin(crop_of_photo[decisions["photo"]], list(TRAIN_CROPS)), "seen by gate",
                             np.where(crop_of_photo[decisions["photo"]] == "tomato", "tomato (should pass)", "unseen"))
reached = ~decisions["outcome"].isin(["too blurred", "not a leaf"])
refused = reached & ~gate_pass[decisions["photo"]]
decisions["after gate"] = np.where(refused, "not our crop (gate)", decisions["outcome"])


def answered(column):
    hit = decisions[column].isin(["ANSWERED", "held: asks to confirm the guessed crop"])
    return decisions.assign(hit=hit).groupby(["gate", "crop_choice"])["hit"].mean()


compare = pd.DataFrame({"answered or held, before": answered("outcome"), "after gate": answered("after gate")})
display((compare * 100).round(1))
print("% of photos the gate refused (of those reaching the disease model), by source:")
display((decisions[reached].assign(refused=refused[reached]).groupby("source")["refused"].mean() * 100).round(1))

## Save the results (and, if adopted, the gate for the backend)

In [ ]:
REPO_RESULTS.mkdir(parents=True, exist_ok=True)
report.to_csv(OUT_DIR / "report.csv")
by_crop.to_csv(OUT_DIR / "unseen_by_crop.csv")
by_source.to_csv(OUT_DIR / "selection_by_source.csv")
compare.to_csv(OUT_DIR / "notebook21d_before_after.csv")
OTHERS.drop(columns=["path"]).to_csv(OUT_DIR / "other_crop_photos.csv", index=False)
summary = {"decided_on": pd.Timestamp.now(tz="UTC").isoformat(), "rerun_reason": RERUN_REASON or None,
           "rules_written": "2026-11-03, before anything was scored", "architecture": ARCHITECTURE,
           "cut_off": CUT_OFF, "selection_passed": selection_pass, "unseen_crops_refused": unseen_reject,
           "needed": {"selection_passed": SELECTION_PASS, "unseen_crops_refused": UNSEEN_REJECT, "pass_rate": PASS_RATE},
           "adopted": ADOPTED, "train_crops": sorted(TRAIN_CROPS), "unseen_crops": sorted(UNSEEN_CROPS | {"bean"}),
           "report": report.reset_index().to_dict(orient="records")}
training.save_json(summary, OUT_DIR / "decision.json")
if ADOPTED:
    leaf_check.export(head, ARCHITECTURE, CUT_OFF, OUT_DIR / "crop_gate.json",
                      {"rule": f"cut-off where {PASS_RATE:.0%} of validation leaves of our crops pass",
                       "selection_passed": selection_pass, "unseen_crops_refused": unseen_reject})
for file in OUT_DIR.glob("*"):
    if file.suffix in {".csv", ".json"}:
        shutil.copy(file, REPO_RESULTS / file.name)
(REPO_RESULTS / "review_sheets").mkdir(exist_ok=True)
for sheet in sheets:
    shutil.copy(sheet, REPO_RESULTS / "review_sheets" / Path(sheet).name)
print(sorted(p.name for p in REPO_RESULTS.iterdir()))
drive.flush_and_unmount()
drive.mount("/content/drive")

## Save to GitHub

If you opened this notebook from GitHub: File -> Download .ipynb, upload it to
MyDrive/Sproutly-app/ml/notebooks (replace), then run the next cell.

In [ ]:
from google.colab import userdata

from src import github_sync

REPO_RESULTS.mkdir(parents=True, exist_ok=True)
shutil.copy(ML_DIR / "notebooks" / "22_crop_gate.ipynb", REPO_RESULTS / "notebook.ipynb")
print(github_sync.push(
    REPO_DIR,
    branch="colab/crop-gate",
    message="Crop gate: our crops or another crop, on the leaf check's frozen network",
    token=userdata.get("GITHUB_TOKEN"),
    name="BilliartMuthoni",
    email=userdata.get("GIT_EMAIL"),
    paths=[str(REPO_RESULTS.relative_to(REPO_DIR))],
))